# Python for YFinance (Yahoo Finance)
完成本 Notebook 后，你应该能够：

- 用 Python 获取金融市场数据
- 完成基础的数据清洗与可视化
- 计算收益率、波动率、移动平均等常见指标
- 设计并回测一个最简单的交易策略
- 对策略表现进行初步评价
- 在教师给出的框架上继续扩展自己的研究思路

In [ ]:
# 如果你本地还没装依赖，可以取消注释运行
# !pip install pandas numpy matplotlib yfinance

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

print('环境准备完成。')



## 1. 选一个你熟悉的资产

这里先从单只股票开始。你可以先用 Apple、Tesla、Microsoft，之后再换成 ETF、银行股、A 股 ADR 等做比较。

思考：

1. 为什么课程一开始先选单资产，而不是一上来就做组合？
2. 对于不同市场、不同资产，数据频率和字段会有什么差别？


In [ ]:

# 你可以修改 ticker 试试不同资产
TICKER = 'AAPL'
START = '2020-01-01'
END = '2026-01-01'

df = yf.download(TICKER, start=START, end=END, auto_adjust=True, progress=False)

# 有些版本会返回多层列名，这里统一处理
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df.head()


In [ ]:

print('数据形状:', df.shape)
print('
列名:')
print(df.columns.tolist())
print('
缺失值统计:')
print(df.isna().sum())



## 2. 价格走势与数据理解

先画最基础的价格图。任何复杂模型之前，先看图，是金融数据分析里非常重要的一步。

思考：

1. 价格图里你能看到哪些阶段性变化？
2. 仅看价格水平图，有哪些信息是看不到的？


In [ ]:

df['Close'].plot(title=f'{TICKER} Adjusted Close Price')
plt.xlabel('Date')
plt.ylabel('Price')
plt.show()



## 3. 计算收益率

在金融研究中，很多分析不是直接用价格，而是用收益率。

下面先给出最常见的两种：
- 简单收益率（simple return）
- 对数收益率（log return）

你需要观察：它们在日频下通常很接近，但在高波动情形下会出现差别。


In [ ]:

df['simple_ret'] = df['Close'].pct_change()
df['log_ret'] = np.log(df['Close'] / df['Close'].shift(1))

df[['Close', 'simple_ret', 'log_ret']].head(10)


In [ ]:

# 课堂练习 1：请你自己补全“累计收益率”列
# 提示：先思考 simple return 和 cumulative return 的关系

def build_cumulative_return(return_series: pd.Series) -> pd.Series:
    "根据日收益率计算累计净值。"
    # TODO: 请把下面这一行替换成你自己的代码
    cum = (1 + return_series.fillna(0)).cumprod()
    return cum

df['cum_nav'] = build_cumulative_return(df['simple_ret'])
df[['cum_nav']].plot(title=f'{TICKER} Cumulative NAV')
plt.show()



## 4. 波动率：风险的最基础度量

波动率通常不是全部风险，但它是最常见、最容易入门的风险指标。

问题：

1. 为什么金融里经常使用“滚动波动率”而不是全样本波动率？
2. 年化波动率为什么要乘上 `sqrt(252)`？


In [ ]:

window = 20

df['rolling_vol_20'] = df['simple_ret'].rolling(window).std() * np.sqrt(252)

df['rolling_vol_20'].plot(title=f'{TICKER} 20-Day Rolling Volatility (Annualized)')
plt.ylabel('Volatility')
plt.show()



## 5. 技术分析入门：移动平均线

参考很多股票分析项目中的常见结构，我们先做最基础的技术指标：短期均线和长期均线。

你可以把它理解成：
- 短期均线更敏感，反映近期价格趋势
- 长期均线更平滑，反映更长期趋势


In [ ]:

short_window = 20
long_window = 60

df['sma_short'] = df['Close'].rolling(short_window).mean()
df['sma_long'] = df['Close'].rolling(long_window).mean()

ax = df[['Close', 'sma_short', 'sma_long']].plot(title=f'{TICKER} Price with Moving Averages')
ax.set_ylabel('Price')
plt.show()


In [ ]:

# 课堂练习 2：自己写一个 EMA（指数移动平均）函数
# 不要直接复制网上代码，先查 pandas 的 ewm 用法，再自己实现

def ema(series: pd.Series, span: int) -> pd.Series:
    # TODO: 请补全
    return series.ewm(span=span, adjust=False).mean()

df['ema_20'] = ema(df['Close'], 20)
df[['Close', 'ema_20']].tail()



## 6. RSI 指标：把“涨跌强弱”量化

很多项目会引入 RSI、MACD、Bollinger Bands 等指标。这里先选 RSI，原因是它既常见，又能训练你写滚动计算逻辑。

任务：

1. 阅读函数结构，理解每一步经济含义
2. 尝试把窗口从 14 改成 6、10、20，比一比差别


In [ ]:

def rsi(series: pd.Series, window: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()

    rs = avg_gain / avg_loss
    rsi_value = 100 - (100 / (1 + rs))
    return rsi_value

df['rsi_14'] = rsi(df['Close'], 14)

df[['rsi_14']].plot(title=f'{TICKER} RSI(14)')
plt.axhline(70, linestyle='--')
plt.axhline(30, linestyle='--')
plt.show()



## 7. 第一个策略：均线交叉策略

这是最经典的教学用策略之一：
- 当短期均线高于长期均线时，持有资产
- 否则空仓

虽然它非常简单，但它足够适合讲清楚：

1. 信号如何定义
2. 仓位如何从信号映射而来
3. 收益如何从仓位与资产收益率得到
4. 为什么回测中要注意“未来函数”


In [ ]:

# 信号：1 表示持有，0 表示空仓
df['signal'] = (df['sma_short'] > df['sma_long']).astype(int)

# 关键：仓位要滞后一日，避免偷看未来
df['position'] = df['signal'].shift(1).fillna(0)

df[['Close', 'sma_short', 'sma_long', 'signal', 'position']].tail(10)


In [ ]:

# 策略收益
df['strategy_ret'] = df['position'] * df['simple_ret']

df['asset_nav'] = (1 + df['simple_ret'].fillna(0)).cumprod()
df['strategy_nav'] = (1 + df['strategy_ret'].fillna(0)).cumprod()

df[['asset_nav', 'strategy_nav']].plot(title=f'{TICKER} Buy&Hold vs SMA Strategy')
plt.ylabel('NAV')
plt.show()


In [ ]:

# 课堂练习 3：请你自己加入交易成本
# 思考：如果每次调仓都有成本，原策略还会那么好吗？

cost_rate = 0.001  # 示例：单边千分之一

def backtest_with_cost(position: pd.Series, asset_ret: pd.Series, cost_rate: float = 0.001) -> pd.Series:
    turnover = position.diff().abs().fillna(0)
    # TODO: 请检查下面的交易成本写法是否合理；也可以自己重写
    strategy_ret = position * asset_ret - turnover * cost_rate
    return strategy_ret

df['strategy_ret_cost'] = backtest_with_cost(df['position'], df['simple_ret'], cost_rate)
df['strategy_nav_cost'] = (1 + df['strategy_ret_cost'].fillna(0)).cumprod()

df[['asset_nav', 'strategy_nav', 'strategy_nav_cost']].plot(title='Impact of Transaction Cost')
plt.show()



## 8. 策略评价：不要只看收益率

下面给出一个最小版绩效评价函数。你应该开始形成一个研究习惯：

任何策略都至少同时看：
- 收益
- 波动
- 夏普比率
- 最大回撤

进一步的问题：

1. 为什么高收益策略不一定是好策略？
2. 夏普比率在什么情况下会误导你？


In [ ]:

def max_drawdown(nav: pd.Series) -> float:
    running_max = nav.cummax()
    drawdown = nav / running_max - 1
    return drawdown.min()


def evaluate_strategy(ret: pd.Series, nav: pd.Series) -> pd.Series:
    ret = ret.dropna()
    ann_return = (1 + ret).prod() ** (252 / len(ret)) - 1 if len(ret) > 0 else np.nan
    ann_vol = ret.std() * np.sqrt(252)
    sharpe = ann_return / ann_vol if pd.notna(ann_vol) and ann_vol != 0 else np.nan
    mdd = max_drawdown(nav.dropna())

    return pd.Series({
        'annual_return': ann_return,
        'annual_volatility': ann_vol,
        'sharpe_ratio': sharpe,
        'max_drawdown': mdd
    })

report = pd.DataFrame({
    'Buy_and_Hold': evaluate_strategy(df['simple_ret'], df['asset_nav']),
    'SMA_Strategy': evaluate_strategy(df['strategy_ret'], df['strategy_nav']),
    'SMA_With_Cost': evaluate_strategy(df['strategy_ret_cost'], df['strategy_nav_cost'])
})

report.T


In [ ]:

# 课堂练习 4：请你自己补一个你认为重要的指标
# 可选方向：胜率、Calmar ratio、Sortino ratio、信息比率等

def win_rate(ret: pd.Series) -> float:
    ret = ret.dropna()
    # TODO: 请补全
    return (ret > 0).mean()

print('Strategy win rate:', win_rate(df['strategy_ret_cost']))



## 9. 从“会跑代码”到“会提出问题”

到这里，真正重要的已经不是 Notebook 本身，而是你能不能提出更好的金融问题。

你可以继续探索：

1. 把单只股票换成 ETF，策略是否更稳定？
2. 把均线交叉换成 RSI 信号，会怎样？
3. 把日频换成周频，结论是否变化？
4. 在危机期和非危机期，策略表现是否不同？
5. 不同参数下是否存在“过拟合”问题？


In [ ]:

# 开放作业模板：多资产比较
# 你需要补全循环、汇总表和可视化

tickers = ['AAPL', 'MSFT', 'GOOGL', 'SPY']
results = []

for ticker in tickers:
    data = yf.download(ticker, start=START, end=END, auto_adjust=True, progress=False)
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)

    # TODO 1: 计算收益率
    # TODO 2: 计算短长期均线
    # TODO 3: 构建 signal / position
    # TODO 4: 计算 strategy_ret 和 strategy_nav
    # TODO 5: 调用 evaluate_strategy

    # 示例占位：请你替换成真实结果
    results.append({'ticker': ticker, 'note': 'replace with your own backtest result'})

pd.DataFrame(results)



## 10. 课程项目建议

你可以把这个 Notebook 的最后一部分扩展成一个小项目。下面给出几个适合作为课程论文或课堂展示的题目：

1. 技术指标比较：SMA、EMA、RSI 哪一种在同一资产上表现更稳健？
2. 资产比较：科技股、价值股、指数 ETF 的策略表现是否有系统差异？
3. 参数敏感性：不同均线窗口下，策略结果变化有多大？
4. 市场状态切换：上涨期、下跌期、震荡期中策略是否表现不同？
5. 加入基本面变量：能否把估值指标或财报变量纳入择时框架？

建议最终提交内容：
- 研究问题
- 数据来源
- 策略定义
- 回测结果
- 风险评价
- 局限性与改进方向



## 11. 给教师的使用建议

这个 Notebook 可以直接作为 1～2 次课的实践主线：

1. 第一节课先带学生完整跑通；
2. 第二节课让学生补全 TODO；
3. 第三步布置开放作业，要求每位学生或每组至少改动一个“资产 + 指标 + 参数”；
4. 展示时不要只讲结果，重点讲“为什么这样设计”。

这样学生会从“照抄代码”逐渐过渡到“提出问题—构造指标—评价策略”的研究型学习方式。
